# `_estimate_sigma` / `_fit_one_profile` rewrite

**What was wrong**

1. `_estimate_sigma` took the second moment of the *whole* profile after subtracting
   only the median. Clipping negatives to zero rectifies the background noise, so
   ~2/3 of the weight was noise spread over +/-512 px. The raw result (~256) always
   hit the `profile.size/4` clip, so the "measurement" was really a constant.
2. `p0_amp = profile.max()` -- but `amp` in the model is height *above* `offset`.
   The guess was ~24x too large (21955 vs 926).
3. `break` accepted the first rung that did not *raise*, not the one that fit best.
   On the railed frames, 5 of 6 rungs found the right answer with a clearly lower
   residual (61 vs 94); rung 0 lost and won anyway.

**What changed**

| | old | new |
|---|---|---|
| sigma guess | 2nd moment of whole profile -> clipped constant | half-max crossing width of the peak |
| amp guess | `profile.max()` (absolute) | `profile.max() - median` (above offset) |
| unmeasurable case | returns `50.0` | returns `None`, caller falls back |
| rung selection | first that doesn't raise | lowest RMS residual |
| diagnostics | none | `resid`, `sigma_est`, `p0_sigma`, `on_bound` |

Constants that remain, and why they are not magic numbers in the old sense:
`SMOOTH_WIN=5` (boxcar for peak finding; its width is removed in quadrature
afterwards) and `MIN_PEAK_SNR=5` (below this the peak is not distinguishable
from noise, so we decline to measure rather than fabricate). The
`FALLBACK_SIGMAS` ladder is coarse bracketing, only reached when the measurement
declines -- and because every candidate is now scored by residual, a bad rung
cannot win.

In [ ]:
import time
import numpy as np, glob
from astropy.io import fits
from scipy.optimize import curve_fit
import fits_reprocess as fr

files = sorted(glob.glob(r'E:/Reverse Telescope Test Data/20260213_data/allmetal/allmetal_fits/*.fits'))
print('n files', len(files))

def gaussian(x, amp, mu, sigma, offset):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2) + offset

FWHM_FACTOR = 2.0 * np.sqrt(2.0 * np.log(2.0))

## New `_estimate_sigma` (+ helpers)

In [ ]:
SMOOTH_WIN   = 5     # boxcar width (px) used ONLY to locate and measure the peak
MIN_PEAK_SNR = 5.0   # peak must clear this many noise sigma to be measurable


def _boxcar(a, w):
    """Moving average with edge padding, same length as input."""
    if w <= 1:
        return a
    pad = w // 2
    return np.convolve(np.pad(a, pad, mode="edge"), np.ones(w) / w, mode="valid")


def robust_noise_sigma(profile):
    """Per-sample noise sigma from the MAD of successive differences.

    Differencing kills any smooth baseline, and the peak occupies only a handful
    of samples, so the median absolute deviation here measures noise, not signal.
    1.4826 converts MAD -> sigma for a normal; /sqrt(2) undoes the differencing.
    """
    d = np.diff(np.asarray(profile, dtype=np.float64))
    mad = float(np.median(np.abs(d - np.median(d))))
    return mad * 1.4826 / np.sqrt(2.0)


def locate_peak(profile):
    """(index, height above baseline, per-sample noise sigma) from a smoothed copy."""
    profile = np.asarray(profile, dtype=np.float64)
    sm = _boxcar(profile, SMOOTH_WIN)
    baseline = float(np.median(sm))
    i0 = int(np.argmax(sm))
    noise = robust_noise_sigma(profile) / np.sqrt(SMOOTH_WIN)
    return i0, float(sm[i0]) - baseline, noise


def _estimate_sigma(profile, x=None, mu_guess=None):
    """Estimate sigma from the half-maximum crossing width around the peak.

    Measured on a lightly smoothed copy so a single noise sample can neither
    define the peak nor truncate the width; the boxcar's own variance is then
    removed in quadrature. Only samples between the two half-max crossings
    influence the answer, so the background pedestal -- which swamped the old
    whole-profile second moment -- carries no weight at all.

    Returns None when the peak is not measurable (too faint, or running off an
    edge) so the caller can fall back explicitly instead of being handed a
    fabricated number. `x` is accepted and ignored, to keep the old signature.
    """
    profile = np.asarray(profile, dtype=np.float64)
    n = profile.size
    if n < 2 * SMOOTH_WIN:
        return None
    sm = _boxcar(profile, SMOOTH_WIN)
    baseline = float(np.median(sm))

    i0, peak, noise = locate_peak(profile)
    # honour a caller-supplied mu only if it really sits on the peak
    if mu_guess is not None and np.isfinite(mu_guess):
        j = int(round(mu_guess))
        if 0 <= j < n and sm[j] - baseline > 0.5 * peak:
            i0, peak = j, float(sm[j]) - baseline

    if not np.isfinite(peak) or peak <= 0.0:
        return None
    if noise > 0 and peak < MIN_PEAK_SNR * noise:
        return None                       # indistinguishable from background

    half = baseline + 0.5 * peak

    li = i0
    while li > 0 and sm[li] > half:
        li -= 1
    if sm[li] > half:
        return None                       # peak runs off the low edge
    ri = i0
    while ri < n - 1 and sm[ri] > half:
        ri += 1
    if sm[ri] > half:
        return None                       # peak runs off the high edge

    dl = sm[li + 1] - sm[li]
    lx = li + ((half - sm[li]) / dl if dl > 0 else 0.0)
    dr = sm[ri - 1] - sm[ri]
    rx = (ri - 1) + ((sm[ri - 1] - half) / dr if dr > 0 else 1.0)

    fwhm = rx - lx
    if not np.isfinite(fwhm) or fwhm <= 0.0:
        return None

    sigma_meas = fwhm / FWHM_FACTOR
    var = sigma_meas ** 2 - (SMOOTH_WIN ** 2 - 1.0) / 12.0   # deconvolve the boxcar
    return float(np.sqrt(var)) if var > 0.25 else 0.5

## New `_fit_one_profile`

In [ ]:
# Coarse bracketing values, only reached when the measurement declines. Safe
# because every candidate below is scored by residual -- a bad rung cannot win.
FALLBACK_SIGMAS = (2.0, 8.0, 32.0, 128.0)


def _fit_one_profile(profile, return_diagnostics=False):
    """Fit a single Gaussian to a 1D profile. Returns (amp, mu, sigma, offset) or NaNs.

    Tries every candidate initial sigma and keeps the fit with the lowest RMS
    residual, rather than the first one that happens not to raise.
    """
    profile = np.asarray(profile, dtype=np.float64)
    n = profile.size
    fail = {"amp": np.nan, "mu": np.nan, "sigma": np.nan, "offset": np.nan,
            "resid": np.nan, "sigma_est": np.nan, "p0_sigma": np.nan,
            "on_bound": False, "n_tried": 0, "n_converged": 0}
    if n < 5 or not np.all(np.isfinite(profile)):
        return fail if return_diagnostics else (np.nan,) * 4

    x        = np.arange(n, dtype=np.float64)
    baseline = float(np.median(profile))
    mu_guess = float(np.argmax(profile))
    amp_guess = float(profile.max()) - baseline    # amp is height ABOVE offset
    if amp_guess <= 0.0:
        return fail if return_diagnostics else (np.nan,) * 4

    sigma_est = _estimate_sigma(profile, x, mu_guess)

    cands = []
    if sigma_est is not None:
        cands += [sigma_est, sigma_est * 2.0, sigma_est * 0.5]  # bracket the measurement
    cands += list(FALLBACK_SIGMAS)

    sig_hi = n / 2.0
    seen, ladder = set(), []
    for s in cands:
        s = float(np.clip(s, 1.0 + 1e-9, sig_hi * 0.999))
        key = round(s, 6)
        if key not in seen:
            seen.add(key)
            ladder.append(s)

    bounds = ([0.0, 0.0, 1.0, -np.inf], [np.inf, float(n), sig_hi, np.inf])
    best, best_resid, best_s, n_conv = None, np.inf, np.nan, 0
    for s in ladder:
        try:
            popt, _ = curve_fit(gaussian, x, profile,
                                p0=[amp_guess, mu_guess, s, baseline],
                                bounds=bounds, maxfev=5000)
        except Exception:
            continue
        n_conv += 1
        resid = float(np.sqrt(np.mean((gaussian(x, *popt) - profile) ** 2)))
        if np.isfinite(resid) and resid < best_resid:
            best, best_resid, best_s = popt, resid, s

    if best is None:
        return fail if return_diagnostics else (np.nan,) * 4

    amp, mu, sigma, offset = (float(v) for v in best)
    sigma = abs(sigma)
    tol = 1e-6
    on_bound = bool(mu <= tol or mu >= n - tol
                    or sigma >= sig_hi - tol or sigma <= 1.0 + tol)

    if return_diagnostics:
        return {"amp": amp, "mu": mu, "sigma": sigma, "offset": offset,
                "resid": best_resid,
                "sigma_est": np.nan if sigma_est is None else sigma_est,
                "p0_sigma": best_s, "on_bound": on_bound,
                "n_tried": len(ladder), "n_converged": n_conv}
    return amp, mu, sigma, offset

## 1. Synthetic sanity check (no disk I/O)

Known sigma, known mu, realistic pedestal (21000) and noise (~60 counts -- the RMS residual seen on the real frames).

In [ ]:
rng = np.random.default_rng(0)
print('%-30s %9s %10s %9s %7s' % ('case', 'est_sig', 'fit_mu', 'fit_sig', 'bound'))
for true_sig in [3., 5., 12., 40.]:
    for noise, label in [(60., 'realistic'), (174., 'harsh')]:
        xx = np.arange(1024.)
        prof = gaussian(xx, 926., 537., true_sig, 21000.) + rng.normal(0, noise, 1024)
        se = _estimate_sigma(prof)
        d  = _fit_one_profile(prof, return_diagnostics=True)
        print('%-30s %9s %10.3f %9.3f %7s' % (
            'true_sig=%-4g noise=%s' % (true_sig, label),
            ('%.2f' % se) if se is not None else 'None',
            d['mu'], d['sigma'], d['on_bound']))

## 2. Pathological inputs

Should decline (NaN / `on_bound`) rather than return a confident wrong answer.

In [ ]:
xx = np.arange(1024.)
cases = [
    ('flat',         np.full(1024, 21000.)),
    ('all zeros',    np.zeros(1024)),
    ('one hot px',   np.r_[np.full(500, 21000.), [99999.], np.full(523, 21000.)]),
    ('peak at edge', gaussian(xx, 926., 2., 5., 21000.)),
    ('two peaks',    gaussian(xx, 926., 300., 5., 21000.) + gaussian(xx, 926., 700., 5., 0.)),
]
for name, p in cases:
    se = _estimate_sigma(p)
    d  = _fit_one_profile(p, return_diagnostics=True)
    print('  %-13s est=%-7s mu=%10.3f sig=%9.3f bound=%-6s conv=%d/%d' % (
        name, ('%.2f' % se) if se is not None else 'None',
        d['mu'], d['sigma'], d['on_bound'], d['n_converged'], d['n_tried']))

## 3. Old vs new on the real frames

Frame 1 is a good fit under both. Frames 11 and 18 are the railed ones (`mu_y -> 0`, `sigma_y -> 512`).

In [ ]:
for idx in [0, 10, 17]:      # frame_num 1 (good), 11 (railed), 18 (railed)
    with fits.open(files[idx]) as h:
        img = np.flip(h[0].data, axis=(0, 1)).astype(float)
    py = np.sum(img, axis=1)
    y  = np.arange(py.size)
    mu_g = float(py.argmax())

    print()
    print('=== frame %d   yprof len %d   median=%.0f  peak above bkg=%.0f'
          % (idx + 1, py.size, np.median(py), py.max() - np.median(py)))
    print('  OLD _estimate_sigma -> %8.2f   (clip ceiling size/4 = %.0f)'
          % (fr._estimate_sigma(py, y, mu_g), py.size / 4))
    print('  NEW _estimate_sigma -> %8.2f' % _estimate_sigma(py, y, mu_g))

    a, m, s, o = fr._fit_one_profile(py)
    print('  OLD fit  -> mu=%10.4f  sigma=%9.4f' % (m, s))
    d = _fit_one_profile(py, return_diagnostics=True)
    print('  NEW fit  -> mu=%10.4f  sigma=%9.4f  resid=%7.1f  p0_sigma=%.2f  bound=%s  (%d/%d converged)'
          % (d['mu'], d['sigma'], d['resid'], d['p0_sigma'], d['on_bound'],
             d['n_converged'], d['n_tried']))

    popt, _ = curve_fit(gaussian, y, py, p0=[py.max(), py.argmax(), 5, np.median(py)])
    print('  TRUTH    -> mu=%10.4f  sigma=%9.4f' % (popt[1], popt[2]))

## 4. Sweep: railing rate, agreement with truth, and wall time

Raise `STEP`/`N_MAX` for a wider sample; reading FITS off E: is the slow part.

In [ ]:
STEP, N_MAX = 25, 500     # every 25th frame of the first 500

rows = []
t_old = t_new = 0.0
for idx in range(0, min(N_MAX, len(files)), STEP):
    with fits.open(files[idx]) as h:
        img = np.flip(h[0].data, axis=(0, 1)).astype(float)
    for prof, tag in ((np.sum(img, axis=0), 'x'), (np.sum(img, axis=1), 'y')):
        v = np.arange(prof.size)

        t0 = time.perf_counter()
        o_amp, o_mu, o_sig, o_off = fr._fit_one_profile(prof)
        t_old += time.perf_counter() - t0

        t0 = time.perf_counter()
        d = _fit_one_profile(prof, return_diagnostics=True)
        t_new += time.perf_counter() - t0

        try:
            tp, _ = curve_fit(gaussian, v, prof,
                              p0=[prof.max(), prof.argmax(), 5, np.median(prof)])
            t_mu = tp[1]
        except Exception:
            t_mu = np.nan

        n = prof.size
        rows.append(dict(
            frame=idx + 1, axis=tag,
            old_railed=bool(o_mu < 1e-3 or o_mu > n - 1e-3 or o_sig > n / 2 - 1e-6),
            new_railed=d['on_bound'],
            old_dmu=abs(o_mu - t_mu), new_dmu=abs(d['mu'] - t_mu)))

import pandas as pd
r = pd.DataFrame(rows)
print('profiles fitted: %d   (%d frames x 2 axes)' % (len(r), len(r) // 2))
print()
print('                       OLD      NEW')
print('  railed fits      %8d %8d' % (r.old_railed.sum(), r.new_railed.sum()))
print('  |mu-truth|>1px   %8d %8d' % ((r.old_dmu > 1).sum(), (r.new_dmu > 1).sum()))
print('  max |mu-truth|   %8.1f %8.1f' % (r.old_dmu.max(), r.new_dmu.max()))
print('  median |mu-truth| %7.2e %8.2e' % (r.old_dmu.median(), r.new_dmu.median()))
print('  fit wall time     %7.2fs %7.2fs' % (t_old, t_new))
print()
bad = r[r.old_railed | r.new_railed]
if len(bad):
    print('frames where either railed:')
    print(bad.to_string(index=False))
else:
    print('no railed fits in this sample under either implementation')

## Dropping this into `fits_reprocess.py`

`_estimate_sigma` keeps its old `(profile, x, mu_guess)` signature (`x` ignored),
and `_fit_one_profile` still returns the same 4-tuple by default, so both are
drop-in. `fits_reprocess_parallel.py` imports `_fill_fit_results` from
`fits_reprocess`, so fixing it here fixes both entry points -- nothing in the
parallel file needs to change.

Worth doing at the same time, since the diagnostics now exist:

- write `resid_x` / `resid_y` and `on_bound_x` / `on_bound_y` into `{run}_frames.csv`
  (call `_fill_fit_results` with `return_diagnostics=True`)
- record `nx` / `ny`, so a railed fit is detectable from the CSV alone -- right now
  you cannot tell `sigma == size/2` without knowing `size`
- reconsider `FWHM_MAX_PX = 1000` (the notebook uses 500); with `on_bound` recorded,
  the gate matters much less